In [1]:
import pandas as pd, numpy as np, json, os, re
import glob


In [4]:

def get_table(path):
    # Load prior intermediate df and tab by re-reading the uploaded JSON again
    with open(path,'r',encoding='utf-8') as f:
        data=json.load(f)
        
    records=[]
    for sid, item in data.items():
        sent=item.get("sentence","")
        for ann in (item.get("SDoH") or []):
            for cat, payload in ann.items():
                rec={"sentence_id":sid, "sentence":sent, "category":cat}
                if isinstance(payload, dict):
                    rec.update(payload)
                records.append(rec)
    df=pd.DataFrame(records)
    
    def bucket_experiencer(x):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "patient"
    
        s = str(x).strip().lower()
    
        # patient variants
        if s in {"patients", "patient", "patients and caregivers", "patients and siblings"}:
            return "patient"
    
        # father/mother (FIXED: lowercase comparisons + common synonyms)
        if s in {"father", "dad", "daddy"}:
            return "father"
        if s in {"mother", "mom", "mommy"}:
            return "mother"
    
        # children
        if s in {"child", "children", "kid", "kids", "son", "daughter"}:
            return "children"
    
        # relatives/siblings
        if s in {"siblings", "sibling", "relatives", "relative", "aunt", "uncle", "cousin"}:
            return "relatives"
    
        # friends
        if s in {"friends", "friend", "peer", "peers"}:
            return "friends"
    
        # grandparents (substring match is fine)
        if any(k in s for k in ["grandparent", "grandma", "grandmother", "grandpa", "grandfather"]):
            return "grandparents"
    
        # step-parent
        if s in {"step-parent", "step parent", "stepparent"}:
            return "step-parent"
    
        # default bucket for caregivers/others
        return "parents/caregiver"
    
    df["experiencer_bucket"]=df["Experiencer"].apply(bucket_experiencer)
    
    target_cols = [
        "financial_status","employment_status","education_status","education_level","healthcare_type",
        "social_family_level","social_church_level","social_government_level",
        "mental_adhd","mental_ocd","mental_ptsd","mental_anxiety","mental_depression","mental_bipolar","mental_general","mental_medication","mental_neuro_condition","mental_sleep_problem",
        # "num_caregivers",
        "smoke_status","substanceuse_status",
        "divorce","loss","physical_abuse","psychological abuse","domestic violence","dcf","abandonment",
        "insurance_type",
        "adherence_medication","adherence_therapy","adherence_other",
        "transplant_knowledge","caregiving_knowledge","medication_knowledge",
        "increase_literacy","increase_social_support","increase_financial_support",
        "concern_level",
        "transportation_vehicle_access","transportation_cost","transportation_distance","transportation_license","transportation_violation"
    ]

    # --- NEW: build tab only from experiencers that appear ---
    present_buckets = (
        df["experiencer_bucket"]
          .dropna()
          .astype(str)
          .unique()
          .tolist()
    )
    tab = pd.DataFrame({"Experiencer": present_buckets})

    for c in target_cols:
        if c not in tab.columns:
            tab[c] = pd.NA
    
    # default recommendation = no
    for idx in tab.index:
        tab.at[idx,"increase_literacy"]="no"
        tab.at[idx,"increase_social_support"]="no"
        tab.at[idx,"increase_financial_support"]="no"
    
    def norm(x):
        if x is None or (isinstance(x,float) and np.isnan(x)):
            return None
        s=str(x).strip()
        return s if s!="" else None
    
    def set_first(row_idx, col, val):
        v=norm(val)
        if v is None: 
            return
        if pd.isna(tab.at[row_idx,col]):
            tab.at[row_idx,col]=v
    
    def pick(row_idx, col, val, pri, normalizer=lambda s:s):
        v=norm(val)
        if v is None:
            return
        v=normalizer(v)
        if v is None:
            return
        cur=tab.at[row_idx,col]
        if pd.isna(cur):
            tab.at[row_idx,col]=v
        else:
            c=normalizer(str(cur))
            if pri.get(v,0) > pri.get(c,0):
                tab.at[row_idx,col]=v
    
    # Mappers
    FIN_SEV={"poverty":3,"constrain":2,"normal":1}
    def map_financial(x):
        s=str(x).strip().lower()
        if s in FIN_SEV:
            return s
        if s in ["pto","unpaid time off","fundraising"]:
            return "constrain"
        if s=="insured":
            return "normal"
        return None
    
    def map_employment(x):
        s=str(x).strip().lower()
        if s in ["employed","unemployed","retired","on leave"]:
            return s
        if s in ["part-time","student","homemaker"]:
            return "employed"
        if s=="looking for":
            return "unemployed"
        if s=="fmla":
            return "on leave"
        return None
    
    EDU_PRI={"current":4,"past":3,"future":2,"none":1}
    def map_edu_status(x):
        s=str(x).strip().lower()
        return s if s in EDU_PRI else None
    
    def map_edu_level(x):
        s=str(x).strip().lower().replace(" ","_")
        if s in ["high_level","occupational","general","childhood"]:
            return s
        if s=="highlevel":
            return "high_level"
        return None
    
    HC_PRI={"medications":1,"surgeries/procedures":2,"clinical visits":3,"counseling":4,"hospital stay":5}
    def map_healthcare(x):
        s=str(x).strip().lower()
        if s in ["medications","surgeries/procedures","clinical visits","counseling"]:
            return s
        if s in ["hospital admit","hospital discharge","rehab","transfer","hospital stay"]:
            return "hospital stay"
        return None
    
    SOC_PRI={"low":1,"medium":2,"high":3}
    def map_social_level(x):
        s=str(x).strip().lower()
        return s if s in SOC_PRI else None
    
    def social_col(stype):
        s=str(stype).strip().lower()
        if s=="family":
            return "social_family_level"
        if s=="church":
            return "social_church_level"
        if s in ["community","non-profit","nonprofit","government"]:
            return "social_government_level"
        return None
    
    SM_PRI={"none":1,"past":2,"current":3}
    def map_smoke(x):
        s=str(x).strip().lower()
        return s if s in SM_PRI else None
    
    SUB_PRI={"none":1,"past":2,"current":3}
    def map_substance(x):
        s=str(x).strip().lower()
        return s if s in SUB_PRI else None
    
    TRAUMA_PRI={"none":1,"past":2,"current":3}
    def trauma_col(tt):
        s=str(tt).strip().lower()
        if s in ["divorce or separation","divorce"]:
            return "divorce"
        if s=="loss":
            return "loss"
        if s=="physical":
            return "physical_abuse"
        if s=="psychological":
            return "psychological abuse"
        if s=="domestic violence":
            return "domestic violence"
        if s=="dcf":
            return "dcf"
        if s in ["abandonment","abandoment"]:
            return "abandonment"
        return None
    
    def trauma_status(ts):
        s=str(ts).strip().lower()
        return s if s in TRAUMA_PRI else None
    MH_PRI = {"none": 1, "past": 2, "current": 3}

    def mental_col(tt):
        s = str(tt).strip().lower()
        s = " ".join(s.split())  # normalize multiple spaces
    
        if s == "adhd":
            return "mental_adhd"
        if s == "ocd": 
            return "mental_ocd"
        if s == "ptsd":
            return "mental_ptsd"
        if s == "anxiety":
            return "mental_anxiety"
        if s == "depression":
            return "mental_depression"
        if s == "bipolar":
            return "mental_bipolar"
        if s == "general":
            return "mental_general"
        if s == "medication":
            return "mental_medication"
        if s in ["neuro condition", "neuro_condition", "neurologic condition", "neurological condition"]:
            return "mental_neuro_condition"
        if s in ["sleep problem", "sleep issues", "sleep disorder", "sleep_problem"]:
            return "mental_sleep_problem"
        return None
    
    def norm_mh_status(st):
        if st is None:
            return None
        s = str(st).strip().lower()
        if s == "future":
            return "current"   # or return None if you want to ignore future
        if s in ["none", "past", "current"]:
            return s
        return None
        
    def map_insurance(x):
        s=str(x).strip().lower()
        return s if s in ["public","private"] else None
    
    ADH_PRI={"none":1,"low":2,"normal":3,"high":4}
    def adh_level(x):
        s=str(x).strip().lower()
        if s=="high": return "high"
        if s=="low": return "low"
        return None
    
    def adh_col(at):
        s=str(at).strip().lower()
        if s=="medication": return "adherence_medication"
        if s=="therapy": return "adherence_therapy"
        if s in ["other","appointments"]: return "adherence_other"
        return None
    
    LIT_PRI={"none":1,"low":2,"normal":3,"high":4}
    def lit_level(x):
        s=str(x).strip().lower()
        if s in ["high","low","normal"]:
            return s
        return None
    
    def lit_col(lt):
        s=str(lt).strip().lower()
        if s=="transplant knowledge": return "transplant_knowledge"
        if s=="caregiving knowledge": return "caregiving_knowledge"
        if s=="medication knowledge": return "medication_knowledge"
        return None
    
    CON_PRI={"light":1,"moderate":2,"intense":3}
    def concern(x):
        s=str(x).strip().lower()
        return s if s in CON_PRI else None
    
    def infer_num_caregivers(living_type):
        if living_type is None or (isinstance(living_type,float) and np.isnan(living_type)):
            return None
        s=str(living_type).strip().lower()
        if s=="with single parent": return 1
        if s=="with both parents": return 2
        return None
    
    def easyhard(x):
        s=str(x).strip().lower()
        return s if s in ["easy","hard"] else None
    
    def cost_from(v):
        return "low" if v=="easy" else ("high" if v=="hard" else None)
    
    def dist_from(v):
        return "short" if v=="easy" else ("long" if v=="hard" else None)
    
    def yesno_from(v):
        return "yes" if v=="easy" else ("no" if v=="hard" else None)
    
    bucket_to_idx={b:i for i,b in enumerate(present_buckets)}
    
    for _, r in df.iterrows():
        i=bucket_to_idx[r["experiencer_bucket"]]
        cat=r["category"]
    
        if cat=="Financial":
            v=map_financial(r.get("FinancialStatus"))
            if v:
                pick(i,"financial_status",v,FIN_SEV,normalizer=lambda s:str(s).lower())
    
        elif cat=="Employment":
            v=map_employment(r.get("EmploymentStatus"))
            if v:
                set_first(i,"employment_status",v)
    
        elif cat=="Education":
            v=map_edu_status(r.get("EducationStatus"))
            if v:
                pick(i,"education_status",v,EDU_PRI,normalizer=lambda s:str(s).lower())
            lv=map_edu_level(r.get("EducationType"))
            if lv:
                set_first(i,"education_level",lv)
    
        elif cat=="Healthcare":
            v=map_healthcare(r.get("HealthcareType"))
            if v:
                pick(i,"healthcare_type",v,HC_PRI,normalizer=lambda s:str(s).lower())
    
        elif cat=="Social":
            col=social_col(r.get("SocialType"))
            lvl=map_social_level(r.get("SocialActivity"))
            if col and lvl:
                pick(i,col,lvl,SOC_PRI,normalizer=lambda s:str(s).lower())
    
        # elif cat=="Living":
        #     n=infer_num_caregivers(r.get("LivingType"))
        #     if n is not None:
        #         cur=tab.at[i,"num_caregivers"]
        #         if pd.isna(cur):
        #             tab.at[i,"num_caregivers"]=int(n)
        #         else:
        #             try:
        #                 tab.at[i,"num_caregivers"]=max(int(cur), int(n))
        #             except:
        #                 tab.at[i,"num_caregivers"]=int(n)
    
        elif cat=="Smoke":
            v=map_smoke(r.get("SmokeStatus"))
            if v:
                pick(i,"smoke_status",v,SM_PRI,normalizer=lambda s:str(s).lower())
    
        elif cat in ["SubstanceUse","Substance Use"]:
            v=map_substance(r.get("SubstanceUseStatus"))
            if v:
                pick(i,"substanceuse_status",v,SUB_PRI,normalizer=lambda s:str(s).lower())
    
        elif cat=="Trauma":
            col=trauma_col(r.get("TraumaType"))
            ts=trauma_status(r.get("TraumaStatus"))
            if col and ts:
                pick(i,col,ts,TRAUMA_PRI,normalizer=lambda s:str(s).lower())
                
        elif cat in ["MentalHealth", "Mental Health"]:
            col = mental_col(r.get("MentalHealthType"))
            v = norm_mh_status(r.get("MentalHealthStatus"))
            if col and v:
                pick(i, col, v, MH_PRI, normalizer=lambda s: str(s).lower())
                
        elif cat=="Insurance":
            v=map_insurance(r.get("InsuranceType"))
            if v:
                set_first(i,"insurance_type",v)
    
        elif cat=="Adherence":
            col=adh_col(r.get("AdherenceType"))
            lv=adh_level(r.get("AdherenceLevel"))
            if col and lv:
                pick(i,col,lv,ADH_PRI,normalizer=lambda s:str(s).lower())
    
        elif cat=="Literacy":
            col=lit_col(r.get("LiteracyType"))
            lv=lit_level(r.get("LiteracyLevel"))
            if col and lv:
                pick(i,col,lv,LIT_PRI,normalizer=lambda s:str(s).lower())
    
        elif cat=="Recommendation":
            rt=str(r.get("RecommendationType") or "").strip().lower()
            if rt in ["education","increase literacy","increase_literacy","increase literacy"]:
                tab.at[i,"increase_literacy"]="yes"
            if rt in ["increase social support","increase_social_support"]:
                tab.at[i,"increase_social_support"]="yes"
            if rt in ["increase financial support","increase_financial_support"]:
                tab.at[i,"increase_financial_support"]="yes"
    
        elif cat=="Concern":
            v=concern(r.get("ConcernLevel"))
            if v:
                pick(i,"concern_level",v,CON_PRI,normalizer=lambda s:str(s).lower())
    
        elif cat=="Transportation":
            tt=str(r.get("TransportationType") or "").strip().lower()
            conv=easyhard(r.get("TransportationConvenienceLevel"))
            if not conv:
                continue
            if tt=="vehicle access":
                set_first(i,"transportation_vehicle_access",conv)
            elif tt=="transportation cost":
                v=cost_from(conv)
                if v:
                    set_first(i,"transportation_cost",v)
            elif tt=="travel distance":
                v=dist_from(conv)
                if v:
                    set_first(i,"transportation_distance",v)
            elif tt=="license":
                v=yesno_from(conv)
                if v:
                    set_first(i,"transportation_license",v)
    return tab

In [16]:
df = pd.DataFrame([])
for path in glob.glob("../output/*.json"):
    tab = get_table(path)
    doc_id = path.split("\\")[-1].split("_")[0]
    tab.insert(0,"doc_id", doc_id)
    df = pd.concat([df,tab])

In [17]:
sorted_df = df.sort_values(by='doc_id', ascending=True).reset_index(drop=True)

In [22]:
sorted_df.employment_status.value_counts()

Series([], Name: count, dtype: int64)

In [164]:
sorted_df.to_csv("../data/curated_data.csv")